## 01_xi_ww — Woolworths 自己相関関数 ξ_WW(r)

**入力**
- `output/data/woolworths_locations.csv` — Woolworths 店舗座標 (N_W = 1,039)
- `output/data/random_masked.csv`        — 共有大陸マスク済みランダムカタログ (N_R = 10,390)

**出力**
- `output/xi/xi_ww.csv`                 — ξ_WW(r): r_km, xi, xi_err, n_rr (38 ビン)

**前提**
- `00_random_catalog.ipynb` 実行済み
- `01_xi_cc.ipynb` 実行済み（ビン設計 rMin を xi_cc.csv の先頭 r から確認可）
- `gradle :lib:jar` 完了

**注意**
- ビン設計は Coles と同一にする（meanNN を Woolworths 座標から再計算するが、
  出力 CSV の r_km が xi_cc.csv と対応していることを確認すること）

In [1]:
@file:DependsOn("../../lib/build/libs/retail-utils-1.0.jar")

In [2]:
%use dataframe
%use lets-plot

import retail.*
import kotlin.math.*

In [3]:
// --- データ読み込み ---
val dfW      = DataFrame.readCSV("./output/data/woolworths_locations.csv")
val dfRandom = DataFrame.readCSV("./output/data/random_masked.csv")

val woolPoints:   List<Point> = dfW.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }
val randomPoints: List<Point> = dfRandom.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }

val nW = woolPoints.size
val nR = randomPoints.size
println("N_W (Woolworths) = $nW")
println("N_R (random)     = $nR")

N_W (Woolworths) = 1039
N_R (random)     = 10390


In [4]:
// --- ビン設計（Woolworths の meanNN を基準）---
// Coles と同じ Δln r = 0.15, rMax = 2000 km
// rMin は Woolworths の meanNN/2 から設計するため Coles とわずかに異なる場合があるが
// 対数ビン幅が同じなら比較は成立する
val nnW   = woolPoints.map { p1 -> woolPoints.filter { it !== p1 }.minOf { haversine(p1, it) } }
val meanNNw = nnW.average()

val bins    = logBins(rMin = meanNNw / 2.0)
val centers = binCenters(bins)
val nBins   = centers.size

println("meanNN (Woolworths) = %.2f km".format(meanNNw))
println("rMin   = %.2f km  (= meanNN / 2)".format(meanNNw / 2.0))
println("nBins  = $nBins")

meanNN (Woolworths) = 12.78 km
rMin   = 6.39 km  (= meanNN / 2)
nBins  = 39


In [5]:
// --- DD / DR / RR ペアカウント ---
println("DD_WW を計算中...")
val nDD = pairCounts(woolPoints, null, bins)

println("DR_WW を計算中...")
val nDR = pairCounts(woolPoints, randomPoints, bins)

println("RR を計算中...")
val nRR = pairCounts(randomPoints, null, bins)

println("完了")

DD_WW を計算中...
DR_WW を計算中...
RR を計算中...
完了


In [6]:
// --- Landy-Szalay 推定量 ---
val normDD = nW.toLong() * (nW - 1) / 2
val normDR = nW.toLong() * nR
val normRR = nR.toLong() * (nR - 1) / 2

val xiWW: List<XiBin> = landySzalay(nDD, nDR, nRR, normDD, normDR, normRR, bins)

println("%-8s  %-9s  %-9s".format("r [km]", "ξ_WW", "σ_ξ"))
println("-".repeat(32))
xiWW.forEach { b -> println("%-8.1f  %-9.4f  %.5f".format(b.rCenter, b.xi, b.xiErr)) }

r [km]    ξ_WW       σ_ξ      
--------------------------------
6.9       236.0668   13.19075
8.0       212.2762   10.16754
9.2       217.6798   9.52584
10.7      187.8567   6.81478
12.4      166.9037   5.18409
14.4      158.5104   4.31582
16.7      148.1831   3.54096
19.3      132.9559   2.75162
22.4      108.5271   1.93709
25.9      89.6974    1.38312
30.0      71.2544    0.95804
34.8      52.3632    0.60252
40.3      37.0610    0.37472
46.7      27.1856    0.24246
54.1      20.0072    0.15654
62.7      15.4410    0.10523
72.7      11.8806    0.07118
84.2      8.7505     0.04697
97.6      5.8390     0.02840
113.0     5.3224     0.02285
131.0     4.2569     0.01645
151.8     3.0538     0.01102
175.9     2.0016     0.00710
203.8     1.5051     0.00516
236.2     2.0053     0.00539
273.6     1.0247     0.00317
317.1     0.7509     0.00239
367.4     0.6956     0.00203
425.8     0.8407     0.00193
493.3     0.5671     0.00145
571.7     0.8874     0.00154
662.4     2.8209     0.00276
767.6 

In [7]:
// --- ξ_WW(r) プロット (log-x) ---
val rVec  = xiWW.map { it.rCenter }
val xiVec = xiWW.map { it.xi }
val xiLo  = xiWW.map { it.xi - it.xiErr }
val xiHi  = xiWW.map { it.xi + it.xiErr }

letsPlot(mapOf("r" to rVec, "xi" to xiVec, "lo" to xiLo, "hi" to xiHi)) +
    geomRibbon(alpha = 0.20, fill = "#E87040") { x = "r"; ymin = "lo"; ymax = "hi" } +
    geomLine(color = "#E87040", size = 1.3) { x = "r"; y = "xi" } +
    geomPoint(color = "#E87040", size = 2.0) { x = "r"; y = "xi" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 71.3,  linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ_WW(r)") +
    ggtitle("Woolworths 自己相関関数 ξ_WW(r)",
            "点線: r_D = 71.3 km | 赤点線: r_BAO = 666 km") +
    ggsize(800, 450)

<path d="M0.0 16.0 L0.0 16.0 L19.151034943049666 50.35090332001806 L38.30206988609925 44.250389016157726 L57.453104829148884 85.92951209156146 L76.60413977219858 114.86122345525763 L95.75517471524822 126.72611536066631 L114.90620965829791 140.94893064547276 L134.0572446013475 161.46755802515545 L153.20827954439713 193.8065203525961 L172.35931448744677 218.63864165160058 L191.5103494304964 242.81030734643457 L210.66138437354604 267.467048462426 L229.81241931659568 287.36236369818596 L248.96345425964532 300.1830800693593 L268.1144892026949 309.4892880479342 L287.2655241457446 315.4047093544009 L306.4165590887943 320.0095301334328 L325.5675940318439 324.0504172217631 L344.7186289748936 327.80411376439497 L363.86966391794306 328.47304353290355 L383.02069886099275 329.846210353095 L402.17173380404245 331.3944618744908 L421.32276874709214 332.7474136448067 L440.47380369014184 333.38600929709764 L459.6248386331913 332.7449013248279 L478.775873576241 334.00399777833366 L497.9269085192906 334.35571886908514 L517.0779434623403 334.4269974652139 L536.22897840539 334.24127999511745 L555.3800133484397 334.59243110386524 L574.5310482914892 334.1819409452645 L593.6820832345387 331.7034474113318 L612.8331181775884 333.2518579464446 L631.9841531206382 334.8528467340128 L651.1351880636879 335.120821355007 L670.2862230067375 335.0377733048678 L689.4372579497871 334.62946020904735 L708.5882928928368 335.4369090028618 L727.7393278358863 335.9994948582759 L727.7393278358863 336.0 L708.5882928928368 335.4379053430006 L689.4372579497871 334.6312253723741 L670.2862230067375 335.0392716846564 L651.1351880636879 335.1223681851065 L631.9841531206382 334.85486288737815 L612.8331181775884 333.25615742583284 L593.6820832345387 331.7105108199482 L574.5310482914892 334.18588008628905 L555.3800133484397 334.5961405786999 L536.22897840539 334.24623420689767 L517.0779434623403 334.43219974820556 L497.9269085192906 334.36185048548776 L478.775873576241 334.0121271141326 L459.6248386331913 332.75871137723857 L440.47380369014184 333.39921816795135 L421.32276874709214 332.7656094436316 L402.17173380404245 331.42270813306294 L383.02069886099275 329.88836808139894 L363.86966391794306 328.5315811676553 L344.7186289748936 327.87687096541424 L325.5675940318439 324.1707662980185 L306.4165590887943 320.1918992641117 L287.2655241457446 315.67433505670266 L268.1144892026949 309.8903684659972 L248.96345425964532 300.8042971191523 L229.81241931659568 288.3224537425846 L210.66138437354604 269.0108155239178 L191.5103494304964 245.264978220287 L172.35931448744677 222.18244319131452 L153.20827954439713 198.76968731570975 L134.0572446013475 168.51767725076684 L114.90620965829791 150.0214796095256 L95.75517471524822 137.78401010820633 L76.60413977219858 128.14377005652486 L57.453104829148884 103.39017223055276 L38.30206988609925 68.6572676717098 L19.151034943049666 76.40192605005325 L0.0 49.797011769616404 Z" fill="rgb(232,112,64)" stroke-width="1.0" fill-opacity="0.2">
 
 
 
 <path d="M0.0 16.0 L0.0 16.0 L19.151034943049666 50.35090332001806 L38.30206988609925 44.250389016157726 L57.453104829148884 85.92951209156146 L76.60413977219858 114.86122345525763 L95.75517471524822 126.72611536066631 L114.90620965829791 140.94893064547276 L134.0572446013475 161.46755802515545 L153.20827954439713 193.8065203525961 L172.35931448744677 218.63864165160058 L191.5103494304964 242.81030734643457 L210.66138437354604 267.467048462426 L229.81241931659568 287.36236369818596 L248.96345425964532 300.1830800693593 L268.1144892026949 309.4892880479342 L287.2655241457446 315.4047093544009 L306.4165590887943 320.0095301334328 L325.5675940318439 324.0504172217631 L344.7186289748936 327.80411376439497 L363.86966391794306 328.47304353290355 L383.02069886099275 329.846210353095 L402.17173380404245 331.3944618744908 L421.32276874709214 332.7474136448067 L440.47380369014184 333.38600929709764 L459.6248386331913 332.7449013248279 L478.775873576241 334.00399777833366 L497.9269085192906 334.35571886908514 L517.0779434

In [8]:
// --- xi_ww.csv 出力 ---
val outPath = "./output/xi/xi_ww.csv"
java.io.File(outPath).bufferedWriter().use { w ->
    w.appendLine("r_km,xi,xi_err,n_rr")
    xiWW.forEach { b -> w.appendLine("${b.rCenter},${b.xi},${b.xiErr},${b.nRR}") }
}
println("保存: $outPath  ($nBins 行)")

保存: ./output/xi/xi_ww.csv  (39 行)
